# Breast cancer — voting ensemble

Part of my daily ML practice: one dataset, one question, one notebook.

Small tabular dataset, three different classifiers — does soft voting beat the individuals?

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, VotingClassifier,
                              IsolationForest)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             classification_report, mean_squared_error,
                             mean_absolute_error, r2_score, silhouette_score)

from sklearn.datasets import load_breast_cancer


## Data

In [ ]:
SEED = 1060
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
df = X.copy(); df["target"] = y
print("shape:", df.shape)
print("positive rate: %.3f" % df["target"].mean())
print(df.iloc[:, :3].describe().round(2).to_string())


shape: (569, 31)
positive rate: 0.627
       mean radius  mean texture  mean perimeter
count       569.00        569.00          569.00
mean         14.13         19.29           91.97
std           3.52          4.30           24.30
min           6.98          9.71           43.79
25%          11.70         16.17           75.17
50%          13.37         18.84           86.24
75%          15.78         21.80          104.10
max          28.11         39.28          188.50


## Ensemble vs individuals

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
lr = LogisticRegression(max_iter=5000)
rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1)
svc = SVC(probability=True, random_state=SEED)
vote = VotingClassifier([("lr", lr), ("rf", rf), ("svc", svc)], voting="soft")
for name, m in [("logreg", lr), ("rf", rf), ("svc", svc), ("voting", vote)]:
    m.fit(X_train, y_train)
    print(f"{name:>7}: accuracy {m.score(X_test, y_test):.4f}")


 logreg: accuracy 0.9510
     rf: accuracy 0.9650
/home/hatch/workspace/ml-notebook-a-day/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
    svc: accuracy 0.9231
/home/hatch/workspace/ml-notebook-a-day/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
 voting: accuracy 0.9650


## Notes

- Voting is at least as good as the best single model — cheap insurance.
- All models are in the same band; the dataset is just easy.